# Global equity momentum and risk screen

Use Tyche's classified universe to select liquid large-cap listings by market, then rank them with transparent price-derived signals.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tyche_client import TycheClient

client = TycheClient(
    os.getenv("TYCHE_BASE_URL", "http://localhost:8000"),
    os.getenv("TYCHE_API_KEY", "tyk_paste_your_key_here"),
)

## Build a cross-market universe

Country codes come from Tyche's market-aware symbol classifier. The market-cap floor keeps this example small enough for interactive analysis.

In [ ]:
markets = ["US", "CA", "JP", "KR", "TW"]
universes = []

for country in markets:
    frame = client.universe(country=country, market_cap_min=10e9, limit=500)
    if not frame.empty:
        universes.append(frame.nlargest(10, "market_cap"))

universe = pd.concat(universes, ignore_index=True).drop_duplicates("ticker")
universe[["ticker", "name", "country", "sector", "industry", "market_cap"]].head(15)

## Fetch prices and calculate comparable signals

The client automatically splits the ticker list into batches of 50. Signals use only close prices returned by Tyche.

In [ ]:
prices = client.price_matrix(universe["ticker"], start="2024-01-01").ffill()
valid = prices.columns[prices.notna().sum() >= 150]
prices = prices[valid].dropna()
returns = prices.pct_change().dropna()

signals = pd.DataFrame({
    "momentum_3m": prices.pct_change(63).iloc[-1],
    "momentum_12m": prices.pct_change(252).iloc[-1],
    "volatility_3m": returns.tail(63).std() * np.sqrt(252),
    "drawdown": prices.iloc[-1] / prices.cummax().iloc[-1] - 1,
}).dropna()

signals = signals.join(universe.set_index("ticker")[["name", "country", "sector", "market_cap"]])
signals.head()

## Rank without hiding the model

Each metric is converted to a cross-sectional percentile. Higher momentum and shallower drawdowns help; higher volatility hurts.

In [ ]:
signals["score"] = (
    signals["momentum_3m"].rank(pct=True)
    + signals["momentum_12m"].rank(pct=True)
    + signals["drawdown"].rank(pct=True)
    + (1 - signals["volatility_3m"].rank(pct=True))
) / 4

columns = ["name", "country", "sector", "score", "momentum_3m", "momentum_12m", "volatility_3m", "drawdown"]
signals.sort_values("score", ascending=False)[columns].head(20).style.format({
    "score": "{:.2f}",
    "momentum_3m": "{:.1%}",
    "momentum_12m": "{:.1%}",
    "volatility_3m": "{:.1%}",
    "drawdown": "{:.1%}",
})

## Compare distributions by market

Group summaries help distinguish a broad regional move from one exceptional company.

In [ ]:
country_summary = signals.groupby("country").agg(
    names=("score", "size"),
    median_score=("score", "median"),
    median_12m_momentum=("momentum_12m", "median"),
    median_volatility=("volatility_3m", "median"),
).sort_values("median_score", ascending=False)
country_summary

In [ ]:
ax = signals.boxplot(column="momentum_12m", by="country", figsize=(9, 5), grid=False)
ax.set_title("12-month momentum distribution by market")
ax.set_ylabel("return")
plt.suptitle("")
plt.show()

This screen is a reproducible research starting point, not an investment recommendation. Add point-in-time fundamentals and transaction-cost assumptions before using it as a strategy.